# 05. Diffusion: DDPM, DDIM, EDM, and DiT

- **Category:** 05_architectures / 05_generative_models
- **Lineage & Citations:** Jascha Sohl-Dickstein et al., 2015 (*Diffusion Probabilistic Models*); Jonathan Ho et al., 2020 (*DDPM*); Jiaming Song et al., 2020 (*DDIM*); Jonathan Ho & Tim Salimans, 2022 (*Classifier-Free Guidance*); William Peebles & Saining Xie, 2023 (*DiT*).
- **Purpose:** Formulates the discrete and continuous forward noising processes, non-Markovian deterministic DDIM reverse sampling, Classifier-Free Guidance (CFG), and DiT adaLN-Zero modulation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Normalizing Flows RealNVP Glow and Spline Flows](04_normalizing_flows_realnvp_glow_and_spline_flows.ipynb) $\to$ `🔗 Downstream Syntheses:` [Diffusion Conditioning ControlNet and IP-Adapter](06_diffusion_conditioning_controlnet_and_ip_adapter.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**
> - **Linear Noise Schedule for High-Resolution Diffusion:** A linear $\beta_t$ schedule leaves non-zero terminal SNR leakage at $t=T$, causing washed-out contrast; cosine or zero-terminal SNR schedules are required.
> - **High CFG Scale Without Dynamic Thresholding:** Guidance scales $w_{\text{cfg}} > 10$ lead to color oversaturation and pixel blowout without percentile clipping.

## Mathematical Foundations & Governing Equations

### 1. Forward Noising & Deterministic DDIM Reversal
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$$
$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t} \hat{\epsilon}}{\sqrt{\bar{\alpha}_t}}, \quad x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1}} \hat{\epsilon}$$

### 2. Classifier-Free Guidance & DiT adaLN-Zero
$$\tilde{\epsilon}(x_t, c) = \epsilon(x_t, \emptyset) + w_{\text{cfg}} (\epsilon(x_t, c) - \epsilon(x_t, \emptyset))$$
$$y = \alpha \odot (\text{LayerNorm}(x) \odot (1 + \gamma) + \beta)$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

x_0 = weights["x_0"]
noise = weights["noise"]
alpha_bar_t = weights["alpha_bar_t"]
alpha_bar_prev = weights["alpha_bar_prev"]
eps_pred = weights["eps_pred"]
x_dit = weights["x_dit"]
gamma = weights["gamma"]
beta_dit = weights["beta_dit"]
alpha_dit = weights["alpha_dit"]
cfg_scale = float(weights["cfg_scale"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, x_0: {list(x_0.shape)}, x_dit: {list(x_dit.shape)}")

In [ ]:
def ddpm_forward_diffuse(x_0, noise, alpha_bar):
    return torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1.0 - alpha_bar) * noise

def ddim_step(x_t, eps_pred, alpha_bar_t, alpha_bar_prev):
    sqrt_ab_t = torch.sqrt(alpha_bar_t)
    sqrt_one_minus_t = torch.sqrt(1.0 - alpha_bar_t)
    x_0_pred = (x_t - sqrt_one_minus_t * eps_pred) / sqrt_ab_t
    sqrt_ab_prev = torch.sqrt(alpha_bar_prev)
    sqrt_one_minus_prev = torch.sqrt(1.0 - alpha_bar_prev)
    x_prev = sqrt_ab_prev * x_0_pred + sqrt_one_minus_prev * eps_pred
    return x_0_pred, x_prev

def classifier_free_guidance(eps_uncond, eps_cond, cfg_scale):
    return eps_uncond + cfg_scale * (eps_cond - eps_uncond)

def adaln_zero_modulation(x, gamma, beta, alpha, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    norm = (x - mean) / torch.sqrt(var + eps)
    return alpha * (norm * (1.0 + gamma) + beta)

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
x_t = ddpm_forward_diffuse(x_0, noise, alpha_bar_t)
err_xt = float(torch.max(torch.abs(x_t - weights["x_t"])))
assert err_xt < 1e-6, f"DDPM x_t error: {err_xt:.2e}"

x_0_pred, x_prev = ddim_step(x_t, eps_pred, alpha_bar_t, alpha_bar_prev)
err_x0 = float(torch.max(torch.abs(x_0_pred - weights["x_0_pred"])))
err_xp = float(torch.max(torch.abs(x_prev - weights["x_prev"])))
assert err_x0 < 1e-6, f"DDIM x_0 error: {err_x0:.2e}"
assert err_xp < 1e-6, f"DDIM x_prev error: {err_xp:.2e}"

cfg_out = classifier_free_guidance(weights["eps_uncond"], weights["eps_cond"], cfg_scale)
err_cfg = float(torch.max(torch.abs(cfg_out - weights["cfg_out"])))
assert err_cfg < 1e-6, f"CFG error: {err_cfg:.2e}"

adaln_out = adaln_zero_modulation(x_dit, gamma, beta_dit, alpha_dit)
err_adaln = float(torch.max(torch.abs(adaln_out - weights["adaln_out"])))
assert err_adaln < 1e-6, f"adaLN-Zero error: {err_adaln:.2e}"
print(f"Parity Verified: x_t={err_xt:.2e}, DDIM Step={err_xp:.2e}, CFG={err_cfg:.2e}, adaLN={err_adaln:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_ddim = benchmark_fn(lambda: ddim_step(x_t, eps_pred, alpha_bar_t, alpha_bar_prev), iters=50, name="DDIM Step")
b_adaln = benchmark_fn(lambda: adaln_zero_modulation(x_dit, gamma, beta_dit, alpha_dit), iters=50, name="DiT adaLN-Zero")
print(format_benchmark_table([b_ddim, b_adaln]))

# Visualization: Forward SNR Decay & DiT adaLN Modulation
alphas = torch.linspace(1.0, 0.01, 100)
snr = alphas / (1.0 - alphas)

fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].plot(range(100), torch.log10(snr).numpy(), color="purple", label="log10(SNR)")
ax[0].set_title("Signal-to-Noise Ratio (SNR) Schedule")
ax[0].set_xlabel("Diffusion Timestep t")
ax[0].legend(fontsize=8)

ax[1].plot(adaln_out[0].detach().numpy(), "o-", color="teal", label="adaLN-Zero Output")
ax[1].set_title("DiT Block Modulation Magnitude")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse vs Corrected Implementation: High CFG Scale Saturation
# Without clipping or thresholding, cfg_scale = 20 causes extreme variance explosions.
high_cfg = classifier_free_guidance(weights["eps_uncond"], weights["eps_cond"], cfg_scale=20.0)
std_high = float(high_cfg.std().item())
std_normal = float(cfg_out.std().item())
print(f"Normal CFG (w=3.0) std dev: {std_normal:.4f}")
print(f"Excessive CFG (w=20.0) std dev: {std_high:.4f}")
assert std_high > 2.0 * std_normal, "Unbounded high CFG scale must amplify latent variance past normal boundaries!"
print("Demonstrated requirement of dynamic thresholding for high guidance scales.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Deterministic Inversion via DDIM:** Non-Markovian ODE formulations permit fast 20-step sampling without stochastic drift.
2. **Classifier-Free Guidance Amplifies Prompts:** Linearly interpolating between unconditional and conditional predictions steers generation.
3. **Identity Initialization in DiT:** adaLN-Zero gates residual connections to zero at step 0, stabilizing deep Transformer pretraining.

---
### Cross-Domain Generalisation Matrix
- **High-Resolution Generative Video:** Sora and Stable Video Diffusion use DiT architectures over spatiotemporal latent tokens.
- **Robotics Trajectory Synthesis:** Diffusion Policy generates continuous multi-modal action trajectories.
- **De Novo Protein Generation:** RFdiffusion constructs physically valid molecular backbones via $SE(3)$-equivariant diffusion.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_diffusion_ddpm_ddim_edm_and_dit.md](../../../markdown/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.md)
- **PyTorch 2.x Module:** [05_diffusion_ddpm_ddim_edm_and_dit.py](../../../pytorch2/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.py)
- **Functional JAX Module:** [05_diffusion_ddpm_ddim_edm_and_dit.py](../../../jax/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.py)
- **Candle Rust Module:** [05_diffusion_ddpm_ddim_edm_and_dit.rs](../../../candle/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.rs)
- **Hardware Kernel:** [05_diffusion_ddpm_ddim_edm_and_dit.rs](../../../kernels/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.rs)
- **Universal Golden SafeTensors:** [05_diffusion_ddpm_ddim_edm_and_dit.safetensors](../../../weights/05_architectures/05_generative_models/05_diffusion_ddpm_ddim_edm_and_dit.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 04. Normalizing Flows RealNVP Glow and Spline Flows](04_normalizing_flows_realnvp_glow_and_spline_flows.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 06. Diffusion Conditioning ControlNet and IP-Adapter](06_diffusion_conditioning_controlnet_and_ip_adapter.ipynb) &rarr;